# Data preprocessing

## Why do predictive models need clean data?

Linear regression computes its outputs using the formula $y=w^Tx+b$.

What happens if the data set contains missing values? What happens if some values in the data set are represented as text?

**What if the attributes in the data set are represented on very different scales?**

Differences in scale directly affect the geometry of the cost function, which in turn naturally influences the speed and stability of convergence. For example:

- $y=w_1x_1+w_2x_2$
- $MSE=\frac{1}{2N}\sum_{i=1}^{N}\big((w_1x_{i,1}+w_2x_{i,2})-y_i\big)^2$

It is useful to examine how differentiating this function with respect to the individual weights affects gradient values:
- $\frac{\partial MSE}{\partial w_1}=\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i-y_i)x_{i,1}$
- $\frac{\partial MSE}{\partial w_2}=\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i-y_i)x_{i,2}$.

Suppose now that $x_{i,1}$ takes values in $[0,1]$, for example the proportion of correct answers, while $x_{i,2}$ takes values in $[0,100000]$, for example monthly income in PLN.

In this situation:
1. A small change in weight $w_2$ causes a very large change in the predicted value $\hat{y}_i$, because it is multiplied by $x_2$, whose value may be as large as 100000. The cost function is therefore extremely sensitive to movement along the $w_2$ axis.
2. Changing $w_1$ by the same amount has only a small effect on $\hat{y}$ because $x_1\in[0,1]$. The cost function will therefore be very flat along the $w_1$ axis.

## Data splitting

Data splitting is a step in a machine-learning pipeline in which the original data set is divided into at least two disjoint subsets for model training and testing. Training data are used exclusively to optimise the model, whereas test data — which are not seen during training — are used to evaluate the model's ability to generalise.

### Simple split

A simple split can be performed using the `train_test_split` function from the `model_selection` module of `scikit-learn`.

An important limitation of a simple random split is that it does not guarantee preservation of proportions with respect to a selected attribute. For example, if a decision attribute contains ten zeros and ten ones, we do not know whether a test subset of size four will contain four zeros or two zeros and two ones, the latter matching the proportion in the original data set.

In practice, however, `scikit-learn` generally preserves similar fractions of the overall distributions of attributes in the subsets unless a specific stratification attribute is supplied.

In [ ]:
from collections import Counter

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
wine_df = pd.read_csv("data/wine.csv")

In [ ]:
wine_df

In [ ]:
wine_train, wine_test = train_test_split(wine_df, test_size=0.2)

In [ ]:
[f"{val}: {(count / len(wine_df)) * 100}" for val, count in Counter(wine_df["quality"]).items()]

In [ ]:
[f"{val}: {(count / len(wine_train)) * 100}" for val, count in Counter(wine_train["quality"]).items()]

In [ ]:
[f"{val}: {(count / len(wine_test)) * 100}" for val, count in Counter(wine_test["quality"]).items()]

### Stratified split

A stratified split differs from a simple split in one key respect: it preserves the original fraction of objects, according to a chosen attribute, in each resulting subset. When splitting a single data frame, the attribute whose proportions should be preserved must be specified explicitly. When splitting `X, y`, the proportions are determined automatically from labels `y`.

In [ ]:
wine_train, wine_test = train_test_split(wine_df, stratify=wine_df["quality"], test_size=0.2)

In [ ]:
[f"{val}: {(count / len(wine_df)) * 100}" for val, count in Counter(wine_df["quality"]).items()]

In [ ]:
[f"{val}: {(count / len(wine_train)) * 100}" for val, count in Counter(wine_train["quality"]).items()]

In [ ]:
[f"{val}: {(count / len(wine_test)) * 100}" for val, count in Counter(wine_test["quality"]).items()]

In [ ]:
y_col = "quality"
X_cols = list(set(wine_df.columns) - set(y_col))

In [ ]:
X = wine_df[X_cols]
y = wine_df[y_col]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
assert len(X) == len(X_train) + len(X_test)

In [ ]:
[f"{val}: {(count / len(y)) * 100}" for val, count in Counter(y).items()]

In [ ]:
[f"{val}: {(count / len(y_train)) * 100}" for val, count in Counter(y_train).items()]

In [ ]:
[f"{val}: {(count / len(y_train)) * 100}" for val, count in Counter(y_train).items()]

## Handling missing values (data cleaning)

Missing values are one of the most common difficulties in machine learning. Predictive models need complete information about the objects they process, both during training and during inference. Incomplete data must therefore be handled either by removing incomplete objects or attributes entirely, or by using data-imputation techniques.

The choice of method depends heavily on the nature of the data set and the pattern of missing values. Ideally, one should compare the effect of all relevant approaches, although limited computational resources may make this impractical. In practice, intuitive rules are often used: rows may be removed when the fraction of missing values is very small, for example around 0.1% of the original data set; an entire attribute may be removed when a large proportion of its values is missing. In the remaining cases, imputation using an appropriate strategy is usually preferable. Common imputation strategies include:

- arithmetic mean or median (for continuous values),
- mode (for categorical data).

Popular approaches also include trained predictors:
- kNN: estimates a missing value from the nearest neighbours among observations with complete values;
- regressor: estimates a missing value using a regression model trained on observations with complete values.

The semantic context of an attribute can also matter. For example, if a missing value concerns a blood-leucocyte level, one might first check the normal population range and then sample a value from an appropriate probability distribution (e.g. a Gaussian distribution) restricted to that range.

### Data cleaning with pandas

In [ ]:
from sklearn.datasets import fetch_california_housing

In [ ]:
data = fetch_california_housing(as_frame=True)["frame"]

In [ ]:
data

The `isnull` method called on a `Series` object returns a Boolean series indicating whether each value in the column is missing.

In [ ]:
data["MedInc"].isnull()

Calling `any` on the resulting data frame allows us to check whether at least one value is true. The `all` method checks whether every value is true.

In [ ]:
data["MedInc"].isnull().any()

By referring to the column axis (`axis=0` in `any`), we can easily determine which columns contain missing values.

In [ ]:
data.isnull().any(axis=0)

At this point the data set looks ideal: every object has complete information and there are no missing values. Let us deliberately "damage" the data by removing some randomly selected values.

**This step is included only for demonstration and would never be performed in a real preprocessing workflow.**

In [ ]:
# example code that removes a few values at random
from random import randint

# minimum and maximum fraction of cells whose values will be removed
min_percent, max_percent = 0.001, 0.003

# randomly choose between 0.1% and 0.3% of the data-frame cells
cells_to_remove = randint(int(data.size * min_percent), int(data.size * max_percent))

# random selection of row and column indices
for _ in range(cells_to_remove):
    row_idx = randint(0, data.shape[0] - 1)  # random row index
    col_idx = randint(0, data.shape[1] - 1)  # random column index

    # remove the randomly selected cell
    data.iat[row_idx, col_idx] = None

The `data` object now contains several missing values.

In [ ]:
data

In [ ]:
data["MedInc"].isnull()

In [ ]:
data["MedInc"].isnull().any()

In [ ]:
data.isnull().any(axis=0)

If a column contains missing values, they can be replaced with a specified value.

In [ ]:
data["MedInc"].fillna(0)

To replace missing values in the original data frame, call `fillna` on the `DataFrame` object and pass a dictionary mapping column names to the replacement values.

In [ ]:
data.fillna({
    "Longitude": 0,
    "Latitude": 100,
})

The `pandas` library provides methods for computing both basic and more advanced statistics. Of particular interest are `mean()` and `median()`, which return the arithmetic mean and median of values in the selected attribute, respectively.

In [ ]:
data["MedHouseVal"].mean()

In [ ]:
data["MedHouseVal"].median()

For the mode, use the `mode()` method. Remember that a distribution may have more than one mode.

In [ ]:
data["Population"].mode().iloc[0]

In such a case, filling missing values can be reduced to using a ready-made function or a text value describing the aggregator: `mean`, `median`, or `mode`.

In [ ]:
data["MedInc"].fillna("mean")

In [ ]:
data["MedInc"].fillna("median")

In [ ]:
data["MedInc"].fillna("mode")

### Data cleaning with scikit-learn

In addition to `pandas`, `scikit-learn` provides a broad set of data-cleaning tools, particularly for missing-value imputation. The `SimpleImputer` class accepts a `strategy` argument specifying the method used to fill missing values:
- `mean`,
- `median`,
- `most_frequent`,
- `constant`.

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
imputer = SimpleImputer(strategy="median")

Before performing imputation, first select the set of appropriate attributes to which the chosen strategy should be applied.

In [ ]:
num_attributes = data.select_dtypes(include=["number"])

In [ ]:
num_attributes

In [ ]:
num_attributes.isnull().any(axis=0)

Calling `fit` on the created object automatically determines the replacement value for each attribute from the supplied data.

In [ ]:
imputer.fit(num_attributes)

The `statistics_` attribute stores the replacement values determined for the missing data according to the selected strategy.

In [ ]:
imputer.statistics_

The output of `transform` is no longer a data frame but a NumPy array. The original data-frame structure can, however, be reconstructed.

In [ ]:
new_num_attributes = imputer.transform(num_attributes)

In [ ]:
new_num_attributes

In [ ]:
new_num_attributes = pd.DataFrame(new_num_attributes, columns=data.columns)

In [ ]:
new_num_attributes

In [ ]:
new_num_attributes.isnull().any(axis=0)

Because the interfaces of `pandas` and `scikit-learn` differ, it is worth paying attention to the conventions of the tool currently being used. In `scikit-learn`, `fit` and `transform` are two separate operations. Calling `fit` means adapting the transformer to the currently supplied data and learning the values or parameters required for the transformation. Calling `transform` then performs the actual transformation and returns a new object. It is therefore important to remember that calling `transform` on a different data set after fitting on unrelated data may produce meaningless results.

## Scaling and value transformation

### Transforming symbolic values

Data sets often contain textual (symbolic) categorical values. Because machine-learning algorithms operate on numerical values, textual data must be converted to a numerical representation. One option is the `OrdinalEncoder` class in the `preprocessing` module of `scikit-learn`.

In [ ]:
from sklearn.datasets import fetch_kddcup99
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [ ]:
data = fetch_kddcup99(as_frame=True)["frame"]

In [ ]:
data

The `scikit-learn` interface also provides the combined `fit_transform` operation, which performs `fit` followed by `transform`.

In [ ]:
encoder = OrdinalEncoder()
labels = encoder.fit_transform(data[["protocol_type"]])

In [ ]:
labels

The numerical values assigned to the individual text categories are stored in `labels`. These values can then be used in the data frame. The mapping from text values to numerical values is available through the `categories_` attribute of the `encoder` object; the array position corresponds to the assigned number.

In [ ]:
encoder.categories_

Learning algorithms recognise patterns represented by numbers. Some algorithms may therefore incorrectly treat the numerical code assigned to `UDP` as being "greater" or more important than the one assigned to `ICMP`. It is therefore often preferable to use **one-hot encoding**, which creates a sparse vector of length $n$ ($n$ is the number of distinct categories) with a single 1 at the position corresponding to the selected category. In `scikit-learn`, this is implemented by the `OneHotEncoder` class.

In [ ]:
encoder = OneHotEncoder()
protocol_onehot = encoder.fit_transform(data[["protocol_type"]])

In [ ]:
protocol_onehot

The result is an object containing a **sparse matrix**, which saves memory. The `toarray` method can be used to convert it to a NumPy array.

In [ ]:
protocol_onehot.toarray()

Replacing the original attribute with a one-hot representation increases the total number of attributes in the data frame: one new attribute is created for each category.

In [ ]:
onehot_df = pd.DataFrame(
    protocol_onehot.toarray(),
    columns=encoder.get_feature_names_out(),
    index=data.index,
)

In [ ]:
onehot_df

In [ ]:
data = data.join(onehot_df)

In [ ]:
data

### Scaling and transforming numerical attributes

Many learning algorithms perform poorly when numerical attributes are expressed on very different scales (for example 0–1 versus 4–290 — we now know why). Several methods can efficiently transform attribute values to a common range.

One of the most common approaches to scaling numerical attributes is **normalisation** (min-max scaling), which maps all values into a specified interval such as 0–1:
$x'=\frac{x-\min(x)}{\max(x)-\min(x)}$.

In `scikit-learn`, normalisation is implemented by the `MinMaxScaler` class in the `preprocessing` module, where the desired output range can be specified.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [ ]:
scaler = MinMaxScaler(feature_range=(-1, 1))
src_bytes_scaled = scaler.fit_transform(data[["src_bytes"]])

In [ ]:
src_bytes_scaled

In [ ]:
src_bytes_scaled.max(), src_bytes_scaled.min()

**Standardisation** centres the data and scales it according to its dispersion:
$x'=\frac{x-u}{s}$, where:
- $u$ — arithmetic mean,
- $s$ — standard deviation.

In `scikit-learn`, standardisation is implemented by the `StandardScaler` class in the `preprocessing` module.

In [ ]:
scaler = StandardScaler()
dst_bytes_scaled = scaler.fit_transform(data[["dst_bytes"]])

In [ ]:
dst_bytes_scaled

Where and when should normalisation or standardisation be used? In practice, the choice is often made experimentally. It is worth remembering that standardisation is sensitive to outliers, which may strongly affect the resulting range of attribute values.

### Symmetry and normality of numerical attributes

Many learning algorithms make assumptions about the distribution of input attributes; in some settings it is desirable for attributes to be approximately normally distributed. The simplest way to inspect an attribute distribution is to visualise a histogram.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

In [ ]:
data = fetch_california_housing(as_frame=True)["frame"]

In [ ]:
data["MedInc"].hist(bins=100)

If the resulting histograms show clear skewness (as above — right-skewness), a transformation may help make the distribution more symmetric or closer to normal.

**Square-root scale**

In [ ]:
np.sqrt(data["MedInc"]).hist(bins=100)

**Logarithmic scale**

In [ ]:
np.log(data["MedInc"]).hist(bins=100)

**Inverse scale**

The inverse scale reverses the order of values so that the smallest value in the original array becomes the largest and the largest becomes the smallest.

$x'=(max_{val}-min_{val})-x$

In [ ]:
values = (data["MedInc"].max() + data["MedInc"].min()) - data["MedInc"]

In [ ]:
plt.hist(values, bins=100)
plt.show()

**Exponential scale**

In [ ]:

np.pow(data["MedInc"], 2).hist(bins=100)

**Box–Cox scale**

The Box–Cox family consists of power transformations controlled by a single parameter $\lambda$:
$y^{(\lambda)}=\begin{cases}\frac{x^{\lambda}-1}{\lambda} & \text{for } \lambda\neq0\\\ln(x) & \text{for } \lambda=0\end{cases}$

When using the `stats` library, the value of $\lambda$ can be selected automatically by optimisation.

In [ ]:
values, _ = stats.boxcox(data["MedInc"])

In [ ]:
plt.hist(values, bins=100)
plt.show()

**Inverse exponential scale**

The inverse exponential scale combines an exponential transformation with reversal of the direction of the data.

In [ ]:
b = 0.5

values = np.exp(-b * data["MedInc"])

In [ ]:
plt.hist(values, bins=100)
plt.show()

## Transformation pipelines

The previous material makes it clear that the order in which preprocessing operations are applied is extremely important. For large data sets requiring extensive transformation, managing the code and the order of operations can become difficult.

`scikit-learn` **pipelines** solve this problem.

Pipelines make it easier to keep code modular by dividing a workflow into smaller stages. They also help prevent information leakage from training data into test data: transformations are learned only from the training data and then the same learned transformations are applied to test or validation data. This structured approach saves development time and reduces the risk of later errors.

#### Using ready-made transformers

A transformation pipeline can be created using the `Pipeline` class. Its constructor accepts a list of processing steps represented as tuples `(name, estimator)`. In many cases, classes provided by `scikit-learn` (for example `SimpleImputer`) are sufficient. The estimator used in a pipeline must provide both `fit` and `transform` methods.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

In [ ]:
num_values_pipeline = Pipeline([
    ("impute_missing_values", SimpleImputer(strategy="mean")),
    ("scale_values", MinMaxScaler()),
])

An alternative is the `make_pipeline` function, which accepts any number of estimator objects. In this case, step names do not need to be specified explicitly.

In [ ]:
from sklearn.pipeline import make_pipeline

In [ ]:
num_values_pipeline = make_pipeline(
    SimpleImputer(strategy="mean"),
    MinMaxScaler(),
)

Applying a pipeline to a data set requires calling `fit` and then `transform`, or using the combined `fit_transform` method.

In [ ]:
num_values_pipeline.fit_transform(data)

In [ ]:
data_preprocessed = pd.DataFrame(
    num_values_pipeline.fit_transform(data),
    columns=num_values_pipeline.get_feature_names_out(),
    index=data.index,
)

In [ ]:
data_preprocessed

#### Pipelines adapted to attribute data types

When a data set contains different types of attributes (for example numerical and symbolic), a pipeline that fills every missing value using an arithmetic mean will be inappropriate. The `ColumnTransformer` class solves this problem: in addition to a name and an estimator, it accepts a list of attribute names to which a given transformation step should be applied.

In [ ]:
data = fetch_kddcup99(as_frame=True)["frame"]

In [ ]:
data[["duration", "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent"]] = data[["duration", "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent"]].astype(int)

In [ ]:
data[["protocol_type", "service", "flag", "labels"]] = data[["protocol_type", "service", "flag", "labels"]].map(lambda x: x.decode("utf-8"))

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
cat_values_pipeline = make_pipeline(
    OrdinalEncoder(handle_unknown="error"),
)

In [ ]:
preprocessing_pipeline = ColumnTransformer([
    ("num_attributes_steps", num_values_pipeline, data.select_dtypes("number").columns),
    ("cat_attributes_steps", cat_values_pipeline, ("protocol_type", "service", "flag", "labels")),
])

As with `make_pipeline`, the `make_column_transformer` function avoids the need to specify an explicit step name. A useful companion is `make_column_selector`, which automatically selects attributes of a specified type.

In [ ]:
from sklearn.compose import make_column_selector, make_column_transformer

In [ ]:
preprocessing_pipeline = make_column_transformer(
    (num_values_pipeline, make_column_selector(dtype_include="number")),
    (cat_values_pipeline, ("protocol_type", "service", "flag", "labels")),
)

In [ ]:
data_preprocessed = pd.DataFrame(
    preprocessing_pipeline.fit_transform(data),
    columns=preprocessing_pipeline.get_feature_names_out(),
    index=data.index,
)

In [ ]:
data_preprocessed